In [1]:
import os
import cv2
import joblib
import pandas as pd
import numpy as np
from mediapipe.tasks.python import vision

import import_ipynb
from main import (
    options, 
    extract_coordinates_from_video, 
    add_smoothed_features, 
    add_repetitions_column, 
    add_velocity_from_smoothed, 
    add_relative_position_features,
    build_single_rep_vector
)
from ml import COLUMNS_TO_DROP

# Baselines derived from your good_form training data to provide context
GOOD_FORM_BASELINES = {
    "max_torso_lean_degrees": 55.0,  # Alert if lean is higher than this
    "max_depth_ratio": 0.12,          # Alert if depth ratio is too high/shallow depending on your coordinate mapping
}

def generate_diagnostics(rep_vector):
    """Provides actionable feedback if features deviate significantly from good reps."""
    feedback = []
    
    lean = rep_vector.get("max_torso_lean_degrees", 0)
    if lean > GOOD_FORM_BASELINES["max_torso_lean_degrees"]:
        feedback.append(f"   ⚠️  Excessive Torso Lean: {lean:.1f}° (Keep chest up to reduce lower back shear stress)")
        
    # Add any additional threshold warnings here based on your geometry/stability metrics
    
    if not feedback:
        feedback.append("   ✨ Solid positioning maintained throughout the movement.")
    return feedback

def batch_process_and_evaluate(input_folder_path="../user_videos/", model_dir="../models/"):
    model_path = os.path.join(model_dir, "squat_classifier_model.pkl")
    scaler_path = os.path.join(model_dir, "squat_scaler.pkl")
    
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        raise FileNotFoundError("Missing machine learning models in storage. Please train the model first.")
        
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    
    valid_exts = (".mp4", ".avi", ".mov", ".mkv")
    if not os.path.exists(input_folder_path):
        os.makedirs(input_folder_path, exist_ok=True)
        return
        
    video_files = [f for f in os.listdir(input_folder_path) if f.lower().endswith(valid_exts)]
    if not video_files:
        print(f"No targets found. Place clips inside '{input_folder_path}'.")
        return

    print(f"🚀 Found {len(video_files)} target videos. Starting biomechanical evaluation...\n")

    for video_file in video_files:
        video_path = os.path.join(input_folder_path, video_file)
        
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        cap.release()
        
        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            raw_rows = extract_coordinates_from_video(video_path, label="inference", landmarker=landmarker)
            
        if not raw_rows:
            print(f"❌ Skipping {video_file}: MediaPipe failed tracking.")
            continue
            
        df = pd.DataFrame(raw_rows)
        df = add_smoothed_features(df)
        df = add_repetitions_column(df)
        df = add_velocity_from_smoothed(df)
        df = add_relative_position_features(df)
        
        rep_df_groups = df[df['rep_number'] > 0]
        if rep_df_groups.empty:
            print(f"⚠️  No repetitions segmented for {video_file}.")
            continue
            
        rep1_frames = rep_df_groups[rep_df_groups['rep_number'] == 1]
        if not rep1_frames.empty:
            bottom_idx_rep1 = rep1_frames['hip_y_smooth'].idxmax()
            ascent_rep1 = rep1_frames.loc[bottom_idx_rep1:]
            rep_1_avg_ascent_vel = abs(ascent_rep1['hip_velocity_y'].mean()) if not ascent_rep1.empty else None
        else:
            rep_1_avg_ascent_vel = None

        print(f"\n📋 Video Report: {video_file}")
        print("=" * 60)
        
        for rep_num, rep_data in rep_df_groups.groupby('rep_number'):
            rep_vector_dict = build_single_rep_vector(
                rep_df=rep_data,
                video_name=video_file,
                rep_num=rep_num,
                rep_1_avg_ascent_vel=rep_1_avg_ascent_vel,
                fps=fps
            )
            
            vector_df = pd.DataFrame([rep_vector_dict])
            features_to_scale = vector_df.drop(columns=[col for col in COLUMNS_TO_DROP if col in vector_df.columns])
            
            scaled_features = scaler.transform(features_to_scale)
            
            # FIX: Use predict_proba combined with the custom 0.35 threshold from training
            good_probability = model.predict_proba(scaled_features)[0][1]
            is_good_form = good_probability >= 0.35
            
            status_str = "✅ GOOD FORM" if is_good_form else "❌ BAD FORM"
            print(f"• Rep #{rep_num}: {status_str} (Confidence: {good_probability*100:.1f}%)")
            print(f"  [Side View: {rep_vector_dict['body_side'].upper()} | Torso Lean: {rep_vector_dict.get('max_torso_lean_degrees', 0):.1f}°]")
            
            # Print feedback diagnostics if something went wrong
            if not is_good_form:
                diagnostics = generate_diagnostics(rep_vector_dict)
                for line in diagnostics:
                    print(line)
            print("-" * 40)


'../edited_datasets\squat_features.csv' already exists. Skipping dataset creation.
Saved engineered features to '../edited_datasets/squat_features_engineered.csv'
Shape: (8308, 33)

--- Dataset Assembly Complete ---
Successfully processed 35 total individual repetitions.
Final Data Matrix Dimensions: (35, 35)
File stored safely at: '../edited_datasets/squat_rep_vectors_ml.csv'
      visibility                    video_name
4613    0.909228  good_no_weights_personal.mp4
4614    0.909237  good_no_weights_personal.mp4
4616    0.909318  good_no_weights_personal.mp4
4617    0.909605  good_no_weights_personal.mp4
4615    0.909863  good_no_weights_personal.mp4
4612    0.910594  good_no_weights_personal.mp4
4610    0.911458  good_no_weights_personal.mp4
4618    0.911580  good_no_weights_personal.mp4
4625    0.911680  good_no_weights_personal.mp4
4624    0.911726  good_no_weights_personal.mp4
Dataset Loaded: 35 repetitions, 18 features.

--- Starting Leak-Proof Cross Validation ---
Fold 1: 1.00

c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The

In [2]:
batch_process_and_evaluate()

🚀 Found 2 target videos. Starting biomechanical evaluation...

Processing: left_girl.mp4...

📋 Video Report: left_girl.mp4
• Rep #1: ✅ GOOD FORM (Confidence: 96.9%)
  [Side View: RIGHT | Torso Lean: 24.0°]
----------------------------------------
• Rep #2: ✅ GOOD FORM (Confidence: 97.9%)
  [Side View: RIGHT | Torso Lean: 33.7°]
----------------------------------------
Processing: right_girl.mp4...

📋 Video Report: right_girl.mp4
• Rep #1: ✅ GOOD FORM (Confidence: 97.8%)
  [Side View: LEFT | Torso Lean: 41.0°]
----------------------------------------
• Rep #2: ✅ GOOD FORM (Confidence: 94.6%)
  [Side View: LEFT | Torso Lean: 45.3°]
----------------------------------------
